### notes
This code needs:
- the quiz result saved as  "responses.csv" (example found on github)
- the file with countries and resources saved as "links.csv" (found on github)

link to quiz: https://form.typeform.com/to/gER6TXzG


In [1]:
import pandas as pd
import numpy as np

#need to map the name of document
responses = pd.read_csv("responses.csv")
responses.head()

links = pd.read_csv("links.csv", skiprows=1)

responses.head()
print(responses.shape)

(1, 26)


In [2]:
#maps the typeform questions to the right variables
column_mapping = {
    "Which generation are you?" : "generation",
    "Which country do you live in?": "cntry",
    "How much time do you spend on the internet on a typical day? Work and personal use - all devices combined":"netustm",
    "How much of the time during the past week was your sleep restless?":"slprl",
    "How much control do you feel you have over your life in general nowadays?": "ctrlife",
    "How is your health in general?":"health",
    "What is your gender assigned at birth?":"gndr",
    "Have you ever been unemployed and seeking work for a period of more than three months?":"uemp3m",
    "How many people do you have with whom you can discuss intimate and personal matters?":"inprdsc"
}

mapped_df = responses.rename(columns=column_mapping)

new_df= mapped_df[list(column_mapping.values())].copy()

print(new_df)


  generation   cntry  netustm  slprl  ctrlife  health  gndr  uemp3m  inprdsc
0  Millenial  Greece        5      3        0       2     3       3        6


In [3]:
# country specific help pages
print(links.head())

  cntry   country                                               link
0   ---       ---                                                ---
1    AL   Albania               https://worldhotlines.org/country/al
2    AT   Austria  https://worldhotlines.org/country/at#category-...
3    BE   Belgium  https://worldhotlines.org/country/be#category-...
4    BG  Bulgaria               https://worldhotlines.org/country/bg


In [8]:
#changes country name to country shortened, then finds specific resource - if cnoutry not in OG dataset link to general website

country_map = dict(zip(links["country"], links["cntry"]))

new_df["cntry"] = new_df["cntry"].map(country_map).fillna("XX")

matching_row = links[links["cntry"] == new_df["cntry"].iloc[0]]
link = matching_row["link"].iloc[0]

In [5]:
print(type(new_df))
print(new_df.shape)
print(new_df.columns)
print(new_df.dtypes)
print(new_df)

<class 'pandas.DataFrame'>
(1, 9)
Index(['generation', 'cntry', 'netustm', 'slprl', 'ctrlife', 'health', 'gndr',
       'uemp3m', 'inprdsc'],
      dtype='str')
generation      str
cntry           str
netustm       int64
slprl         int64
ctrlife       int64
health        int64
gndr          int64
uemp3m        int64
inprdsc       int64
dtype: object
  generation cntry  netustm  slprl  ctrlife  health  gndr  uemp3m  inprdsc
0  Millenial    GR        5      3        0       2     3       3        6


In [ ]:
#run prediction
import joblib

model = joblib.load("models/mental_health_model.joblib")

probabilities = model.predict_proba(new_df)
probabilities.reshape(1,-1)
risk = probabilities[0, 1] * 100

### personalized outputs
saved in results
depending on prediction, aswell as singular important values like sleep restlessness, life control and health

In [9]:
results = f"We have evaluated your quiz: \n\n People that answer like you have a {risk:.1f}% chance of reporting depressive feelings. Through analyzing your answers we have evaluated some things you could do.\n"
#low risk prediction, still gives info about possible factors + educational resources
if risk <25:
    results += "People that answer like you are predicted to have a low chance of reporting depressive feelings."
    if new_df["ctrlife"].iloc[0]<6 or new_df["slprl"].iloc[0]>=2 or new_df["health"].iloc[0]>=3 or new_df["flt_dpr"].iloc[0] >= 3:
        results += "Here are some additional things you could do to help your mental wellbeing: \n\n"
        if new_df["ctrlife"].iloc[0]<6:
            results += "Percieved control over one's life is a big factor for mental health.At least 45 percent of people that report similar or lower numbers on life control than you also report depressive feelings.  \n\n"
        if new_df["slprl"].iloc[0]>=2:
            results += "Sleep is a big factor for mental well-being. If you feel sad or depressed improving your sleep might give a small benefit to your mental health."
        if new_df["health"].iloc[0]==3:
            results += "Physical health and the perception of it can have major effects on your mental health. More than 50percent of people that evaluated their health similarly to you also report depressive feelings."
        elif new_df["health"].iloc[0]== 4 or new_df["health"].iloc[0]== 5:
            results += "Physical health and the perception of it can have major effects on your mental health. More than 75percent of people that evaluated their health similarly to you also report depressive feelings."
    else:
        results += "You seem to take good care of your mental health. Here are some additional educational resources: \n\n Mental health foundation: \n https://www.mentalhealth.org.uk/supporting-someone-else-their-mental-health \n International mental health association: \n https://imha.ngo/welcome \n Mental health first Aid: \n https://mhfainternational.org/ \n Young People’s Mental Health: \n https://www.ypmh.org/" 
    results += f"If you feel like you are an immediate danger to yourself or others, please reach out to a qualified person. Possible resources can be found under this link: {link}"
# moderate risk, gives info about possible factors, country specific mental health hotlines and general resources        
elif risk >= 25 and risk <= 60:
    results += " People that answer like you are predicted to have a moderate risk of reporting depressive feelings."
    if new_df["ctrlife"].iloc[0]<6 or new_df["slprl"].iloc[0]>=2 or new_df["health"].iloc[0]>=3 or new_df["flt_dpr"].iloc[0] >= 3:
        results += "Here are some additional things you could do to help your mental wellbeing: \n\n"
        if new_df["ctrlife"].iloc[0]<6:
            results += "Percieved control over one's life is a big factor for mental health.At least 45 percent of people that report similar or lower numbers on life control than you also report depressive feelings.  \n\n"
        if new_df["slprl"].iloc[0]>=2:
            results += "Sleep is a big factor for mental well-being. If you feel sad or depressed improving your sleep might give a small benefit to your mental health."
        if new_df["health"].iloc[0]==3:
            results += "Physical health and the perception of it can have major effects on your mental health. More than 50percent of people that evaluated their health similarly to you also report depressive feelings."
        elif new_df["health"].iloc[0]== 4 or new_df["health"].iloc[0]== 5:
            results += "Physical health and the perception of it can have major effects on your mental health. More than 75percent of people that evaluated their health similarly to you also report depressive feelings."
    else:
            results += "You seem to take good care of your mental health. Here are some additional educational resources: \n\n Mental health foundation: \n https://www.mentalhealth.org.uk/supporting-someone-else-their-mental-health \n International mental health association: \n https://imha.ngo/welcome \n Mental health first Aid: \n https://mhfainternational.org/ \n Young People’s Mental Health: \n https://www.ypmh.org/" 
    results += f"If you feel like you are an immediate danger to yourself or others, please reach out to a qualified person. Possible resources can be found under this link: {link}"
#high risk prediction, encourages professional help, gives info about possible factors, emergency resources and general information        
elif risk >60: 
    results += "People that answer like you have a high rate of reporting depressive feelings. We encourage you to reach out to a medical professional."
    results += "Here are some additional things you could do to help your mental wellbeing: \n\n"
    if new_df["ctrlife"].iloc[0]<6:
        results += "Percieved control over one's life is a big factor for mental health.At least 45 percent of people that report similar or lower numbers on life control than you also report depressive feelings.  \n\n"
    if new_df["slprl"].iloc[0]>=2:
        results += "Sleep is a big factor for mental well-being. If you feel sad or depressed improving your sleep might give a small benefit to your mental health."
    if new_df["health"].iloc[0]==3:
        results += "Physical health and the perception of it can have major effects on your mental health. More than 50percent of people that evaluated their health similarly to you also report depressive feelings."
    elif new_df["health"].iloc[0]== 4 or new_df["health"].iloc[0]== 5:
        results += "Physical health and the perception of it can have major effects on your mental health. More than 75percent of people that evaluated their health similarly to you also report depressive feelings."
    results += f"If you feel depressed, please reach out to friends and family and ask for support to get professional help.\n\n If you feel like you are an immediate danger to yourself or others, please reach out to a qualified person. Possible resources can be found under this link: {link} \n\n"
    results += "Here are some additional educational resources: \n\n Mental health foundation: \n https://www.mentalhealth.org.uk/supporting-someone-else-their-mental-health \n International mental health association: \n https://imha.ngo/welcome \n Mental health first Aid: \n https://mhfainternational.org/ \n Young People’s Mental Health: \n https://www.ypmh.org/" 
 
print(results)

We have evaluated your quiz: 

 People that answer like you have a 67.7% chance of reporting depressive feelings. Through analyzing your answers we have evaluated some things you could do.
People that answer like you have a high rate of reporting depressive feelings. We encourage you to reach out to a medical professional.Here are some additional things you could do to help your mental wellbeing: 

Percieved control over one's life is a big factor for mental health.At least 45 percent of people that report similar or lower numbers on life control than you also report depressive feelings.  

Sleep is a big factor for mental well-being. If you feel sad or depressed improving your sleep might give a small benefit to your mental health.If you feel depressed, please reach out to friends and family and ask for support to get professional help.

 If you feel like you are an immediate danger to yourself or others, please reach out to a qualified person. Possible resources can be found under th